# Statistics, tables, figures and LaTeX macros

Aggregate benchmark results into tables, figures, statistics and LaTeX macros.

## Setup
Loads `00_config` (paths) and the library notebooks from this folder.

In [20]:
# Load other notebooks in this folder as modules (all notebooks live in one folder).
import os, sys, types
import nbformat

NB_DIR = os.getcwd()   # open Jupyter in the notebooks folder


def import_notebook(name, filename):
    """Execute the code cells of a notebook in this folder into a module named `name`."""
    if name in sys.modules:
        return sys.modules[name]
    path = os.path.join(NB_DIR, filename)
    mod = types.ModuleType(name)
    mod.__file__ = path
    sys.modules[name] = mod
    for cell in nbformat.read(path, as_version=4).cells:
        if cell.cell_type == "code":
            exec(compile(cell.source, path, "exec"), mod.__dict__)
    return mod


config = import_notebook("config", "00_config.ipynb")

Reads `OUTPUT_DIR/results` and writes everything the manuscript uses to `OUTPUT_DIR/paper_generated`. Copy that folder's contents into `paper/generated/` before compiling the LaTeX manuscript.

In [21]:
import glob
import json
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from scipy.stats import friedmanchisquare, wilcoxon  # noqa: E402
RES = config.RESULTS_DIR
GEN = config.PAPER_GEN_DIR
os.makedirs(GEN, exist_ok=True)
DS = ["CKD", "Heart", "HeartFailure", "BreastCancer", "Cirrhosis", "SUPPORT2", "Diabetes130"]
DSLAB = {"CKD": "CKD", "Heart": "Heart (Cleveland)", "HeartFailure": "Heart failure", "BreastCancer": "Breast cancer",
         "Cirrhosis": "Cirrhosis (PBC)", "SUPPORT2": "SUPPORT2", "Diabetes130": "Diabetes 130"}
MODELS = ["LR", "EBM", "XGB", "MLP"]
NATIVE = {"Exact-linear": "Native", "EBM-native": "Native", "TreeSHAP": "Native"}
EXPL = ["KernelSHAP", "LIME", "Native", "Random"]
COL = {"KernelSHAP": "#2a78d6", "LIME": "#eb6834", "Native": "#1baf7a", "Random": "#8a8985",
       "TreeSHAP": "#1baf7a", "Exact-linear": "#1baf7a", "EBM-native": "#1baf7a"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"font.size": 8, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
                     "font.family": "DejaVu Sans", "axes.titlesize": 8.5, "legend.frameon": False})

### `load` (function)

In [22]:
def load(fname):
    fs = [os.path.join(RES, d, fname) for d in DS if os.path.exists(os.path.join(RES, d, fname))]
    return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()

### `boot_mean` (function)

In [23]:
def boot_mean(x, n=1000, seed=0):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    rng = np.random.default_rng(seed)
    m = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(n)]
    return x.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)

### `tex` (function)

In [24]:
def tex(df, name, caption, label, colfmt=None, size="\\footnotesize", notes=None, wide=False):
    env = "table*" if wide else "table"
    colfmt = colfmt or "l" + "c" * (df.shape[1] - 1)
    L = [rf"\begin{{{env}}}[!htbp]\centering{size}", rf"\caption{{{caption}}}\label{{{label}}}",
         r"\begin{adjustbox}{max width=\linewidth}", rf"\begin{{tabular}}{{{colfmt}}}", r"\toprule", " & ".join(df.columns) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        L.append(" & ".join(str(v) for v in r.values) + r" \\")
    L += [r"\bottomrule", r"\end{tabular}", r"\end{adjustbox}"]
    if notes:
        L.append(rf"\par\smallskip\parbox{{0.95\linewidth}}{{\scriptsize {notes}}}")
    L.append(rf"\end{{{env}}}")
    open(os.path.join(GEN, name), "w").write("\n".join(L) + "\n")

In [25]:
macros = {}

### `M` (function)

In [26]:
def M(name, val, d=3):
    macros[name] = f"{val:.{d}f}" if isinstance(val, (float, np.floating)) else str(val)

## Table 1: datasets

In [27]:
# ---------------------------------------------------------------- Table 1: datasets
metas = {d: json.load(open(os.path.join(RES, d, "meta.json"))) for d in DS if os.path.exists(os.path.join(RES, d, "meta.json"))}
t1 = []
for d, m in metas.items():
    t1.append({"Dataset": DSLAB[d], "Outcome": m["task"], "$n$": f"{m['n']:,}", "Features (cat.)":
               f"{m['n_features']} ({m['n_categorical']})", "Prevalence (\\%)": f"{100 * m['prevalence']:.1f}",
               "Missing (\\%)": f"{100 * m['missing_rate']:.1f}", "Max $|r|$": f"{m['max_abs_corr']:.2f}",
               "$n_{\\text{expl}}$": m["n_explain"]})
tex(pd.DataFrame(t1), "tab_datasets.tex",
    "Datasets. Missing: share of missing raw feature values; Max $|r|$: largest absolute Pearson correlation between "
    "two encoded features in the training split; $n_{\\text{expl}}$: test instances explained.", "tab:datasets",
    colfmt="llrcrrrr", wide=True)
M("nDatasets", len(metas))
M("nTotal", f"{sum(m['n'] for m in metas.values()):,}")

## Table 2: performance

In [28]:
# ---------------------------------------------------------------- Table 2: performance
perf = load("performance.csv")
rows = []
for d in DS:
    r = {"Dataset": DSLAB[d]}
    for k in MODELS:
        p = perf[(perf.dataset == d) & (perf.model == k)]
        if len(p):
            p = p.iloc[0]
            r[k] = f"{p.auroc:.3f} ({p.auroc_lo:.2f}--{p.auroc_hi:.2f})"
    rows.append(r)
tex(pd.DataFrame(rows), "tab_performance.tex",
    "Test-set AUROC (95\\% bootstrap CI) on the held-out 30\\% split used for explanation. Repeated-split means and "
    "AUPRC/Brier scores are given in Table~\\ref{tab:perf2}.", "tab:performance", wide=True)
rows = []
for d in DS:
    for k in MODELS:
        p = perf[(perf.dataset == d) & (perf.model == k)]
        if len(p):
            p = p.iloc[0]
            rows.append({"Dataset": DSLAB[d], "Model": k, "AUROC (5 splits)": f"{p.auroc_rep_mean:.3f} $\\pm$ {p.auroc_rep_sd:.3f}",
                         "AUPRC": f"{p.auprc:.3f}", "Brier": f"{p.brier:.3f}"})
tex(pd.DataFrame(rows), "tab_perf2.tex", "Discrimination over five stratified 70/30 splits (mean $\\pm$ SD) and "
    "AUPRC/Brier score on the explanation split.", "tab:perf2", colfmt="llccc", size="\\scriptsize")
M("aucMin", perf.auroc.min()); M("aucMax", perf.auroc.max())
M("aucMin", perf.auroc.min()); M("aucMax", perf.auroc.max())
M("aucMinDs", DSLAB[perf.loc[perf.auroc.idxmin(), "dataset"]])
gb = []
for d in DS:
    q = perf[perf.dataset == d].set_index("model").auroc
    gb.append(q.max() - max(q.get("LR", 0), q.get("EBM", 0)))
M("glassGapMax", float(max(gb)))

## Instance metrics

In [29]:
# ---------------------------------------------------------------- instance metrics
im = load("instance_metrics.csv")
im["family"] = im.explainer.map(lambda a: NATIVE.get(a, a))
im["gap"] = im.morf - im.lerf
blocks = im.groupby(["dataset", "model", "family"]).agg(gap=("gap", "mean"), morf=("morf", "mean"),
                                                        lerf=("lerf", "mean"), pert_sp=("pert_sp", "mean"),
                                                        pert_jac=("pert_jac", "mean"), retrain_sp=("retrain_sp", "mean"),
                                                        retrain_jac=("retrain_jac", "mean"),
                                                        entropy=("entropy", "mean")).reset_index()
blocks.to_csv(os.path.join(GEN, "blocks.csv"), index=False)
# share of instances where the explainer's MoRF beats the random ranking for the same instance
rnd = im[im.family == "Random"][["dataset", "model", "inst", "morf", "gap"]].rename(columns={"morf": "morf_r", "gap": "gap_r"})
mm = im[im.family != "Random"].merge(rnd, on=["dataset", "model", "inst"])
mm["beats"] = mm.gap > mm.gap_r
beat = mm.groupby("family").beats.mean()
for f_, v in beat.items():
    M(f"beatRandom{f_}", 100 * v, 1)
# Table 3: faithfulness/stability/complexity by model x explainer (mean over datasets of block means)
rows = []
for k in MODELS:
    for f_ in EXPL:
        b = blocks[(blocks.model == k) & (blocks.family == f_)]
        if not len(b):
            continue
        name = f_ if f_ != "Native" else {"LR": "Exact linear", "EBM": "EBM terms", "XGB": "TreeSHAP"}[k]
        rows.append({"Model": k, "Explainer": name, "MoRF $\\uparrow$": f"{b.morf.mean():.3f}",
                     "LeRF $\\downarrow$": f"{b.lerf.mean():.3f}", "Gap $\\uparrow$": f"{b.gap.mean():.3f}",
                     "Stab.\\ input $\\uparrow$": "--" if f_ == "Random" else f"{b.pert_sp.mean():.3f}",
                     "Stab.\\ retrain $\\uparrow$": "--" if f_ == "Random" else f"{b.retrain_sp.mean():.3f}",
                     "Top-5 Jacc.\\ retrain $\\uparrow$": "--" if f_ == "Random" else f"{b.retrain_jac.mean():.3f}",
                     "Entropy $\\downarrow$": f"{b.entropy.mean():.3f}"})
tex(pd.DataFrame(rows), "tab_quality.tex",
    "Explanation quality by model and explainer, averaged over the seven datasets (each dataset first averaged over "
    "its explained instances). MoRF/LeRF: normalised area under the absolute prediction change when features are "
    "deleted most-/least-relevant first; Gap = MoRF $-$ LeRF. Stab.: Spearman correlation of $|$attributions$|$ "
    "under 5\\% input noise (input) or after a bootstrap refit (retrain). Entropy: normalised Shannon entropy of "
    "$|$attributions$|$ (lower = more concentrated). Random: average of five random attribution vectors.",
    "tab:quality", colfmt="llccccccc", wide=True, size="\\scriptsize")

### `friedman` (function)

In [30]:
# Friedman + Nemenyi (LR, EBM, XGB blocks: all four families present)
def friedman(metric, fams):
    b = blocks[blocks.model.isin(["LR", "EBM", "XGB"]) & blocks.family.isin(fams)]
    w = b.pivot_table(index=["dataset", "model"], columns="family", values=metric)[fams].dropna()
    stat, p = friedmanchisquare(*[w[c] for c in fams])
    ranks = w.rank(axis=1, ascending=False).mean()
    q = {3: 2.343, 4: 2.569}[len(fams)]
    cd = q * np.sqrt(len(fams) * (len(fams) + 1) / (6 * len(w)))
    return dict(metric=metric, N=len(w), chi2=stat, p=p, cd=cd, **{f"rank_{c}": ranks[c] for c in fams})

In [31]:
fr = [friedman("gap", EXPL), friedman("morf", EXPL), friedman("pert_sp", EXPL[:3]), friedman("retrain_sp", EXPL[:3]),
      friedman("retrain_jac", EXPL[:3])]
fr = pd.DataFrame(fr)
fr.to_csv(os.path.join(GEN, "friedman.csv"), index=False)
lab = {"gap": "Deletion gap", "morf": "MoRF", "pert_sp": "Input stability", "retrain_sp": "Retrain stability",
       "retrain_jac": "Retrain top-5 Jaccard"}
rows = []
for _, r in fr.iterrows():
    rows.append({"Metric": lab[r.metric], "$N$": int(r.N), "$\\chi^2_F$": f"{r.chi2:.2f}",
                 "$p$": f"{r.p:.2g}" if r.p >= 1e-4 else "$<10^{-4}$",
                 "KernelSHAP": f"{r.rank_KernelSHAP:.2f}", "LIME": f"{r.rank_LIME:.2f}",
                 "Native": f"{r.rank_Native:.2f}",
                 "Random": f"{r.rank_Random:.2f}" if not np.isnan(r.get("rank_Random", np.nan)) else "--",
                 "CD": f"{r.cd:.2f}"})
tex(pd.DataFrame(rows), "tab_friedman.tex",
    "Friedman tests across dataset$\\times$model blocks (LR, EBM, XGB; $N$ = blocks) and mean ranks (1 = best). "
    "Two explainers differ at $\\alpha=0.05$ (Nemenyi) when their mean ranks differ by more than the critical "
    "difference (CD).", "tab:friedman", colfmt="lccccccccc", size="\\scriptsize", wide=True)
for _, r in fr.iterrows():
    key = {"gap": "Gap", "morf": "Morf", "pert_sp": "Pert", "retrain_sp": "Retrain", "retrain_jac": "RetrainJac"}[r.metric]
    M(f"friedP{key}", f"={r.p:.2g}" if r.p >= 1e-4 else "<10^{-4}")
    M(f"friedChi{key}", r.chi2, 2)
    M(f"friedCD{key}", r.cd, 2)
    for c in ["KernelSHAP", "LIME", "Native", "Random"]:
        if f"rank_{c}" in r and not np.isnan(r[f"rank_{c}"]):
            M(f"rank{key}{c}", r[f"rank_{c}"], 2)
# Wilcoxon KernelSHAP vs LIME over all 28 blocks
wil = []
for metric in ["gap", "pert_sp", "retrain_sp", "entropy"]:
    w = blocks[blocks.family.isin(["KernelSHAP", "LIME"])].pivot_table(index=["dataset", "model"], columns="family",
                                                                       values=metric).dropna()
    s = wilcoxon(w.KernelSHAP, w.LIME)
    wins = int((w.KernelSHAP > w.LIME).sum())
    wil.append(dict(metric=metric, n=len(w), p=s.pvalue, ks_better=wins, med_diff=float((w.KernelSHAP - w.LIME).median())))
    key = {"gap": "Gap", "pert_sp": "Pert", "retrain_sp": "Retrain", "entropy": "Ent"}[metric]
    M(f"wilP{key}", f"={s.pvalue:.2g}" if s.pvalue >= 1e-4 else "<10^{-4}")
    M(f"wilWins{key}", wins); M(f"wilN{key}", len(w))
    M(f"wilMed{key}", float((w.KernelSHAP - w.LIME).median()))
pd.DataFrame(wil).to_csv(os.path.join(GEN, "wilcoxon_ks_lime.csv"), index=False)
# headline means
for f_ in ["KernelSHAP", "LIME", "Native", "Random"]:
    b = blocks[blocks.family == f_]
    M(f"gap{f_}", b.gap.mean()); M(f"morf{f_}", b.morf.mean())
    if f_ != "Random":
        M(f"pert{f_}", b.pert_sp.mean()); M(f"retrain{f_}", b.retrain_sp.mean())
        M(f"retrainJac{f_}", b.retrain_jac.mean()); M(f"ent{f_}", b.entropy.mean())
for k in MODELS:
    b = blocks[(blocks.model == k) & (blocks.family != "Random")]
    M(f"retrainModel{k}", b.retrain_sp.mean()); M(f"retrainJacModel{k}", b.retrain_jac.mean())
M("pertDpMax", im.pert_dp.max(), 4); M("pertDpMean", im.pert_dp.mean(), 4)
M("pertDpMax", im.pert_dp.max(), 4); M("pertDpMean", im.pert_dp.mean(), 4)

## Agreement

In [32]:
# ---------------------------------------------------------------- agreement
ag = load("agreement.csv")
ag["pairf"] = ag.apply(lambda r: " vs ".join(sorted([NATIVE.get(r.a, r.a), NATIVE.get(r.b, r.b)])), axis=1)
rows = []
for k in MODELS:
    for pf, g in ag[ag.model == k].groupby("pairf"):
        rows.append({"Model": k, "Pair": pf.replace("Native", {"LR": "Exact linear", "EBM": "EBM terms",
                                                               "XGB": "TreeSHAP", "MLP": "-"}[k]),
                     "Top-5 feature agr.": f"{g.feat_agree.mean():.3f}", "Top-5 sign agr.": f"{g.sign_agree.mean():.3f}",
                     "Rank corr.": f"{g.rank_corr.mean():.3f}",
                     "Range of top-5 agr. over datasets": f"{g.feat_agree.min():.2f}--{g.feat_agree.max():.2f}"})
tex(pd.DataFrame(rows), "tab_agreement.tex",
    "Agreement between explainers on the same model and instances (mean over datasets). Top-5 feature agreement: "
    "share of the five highest-$|$attribution$|$ features shared by both explainers; sign agreement additionally "
    "requires the same sign; rank corr.: Spearman correlation of $|$attributions$|$.", "tab:agreement",
    colfmt="llcccc", size="\\scriptsize")
for pf, g in ag.groupby("pairf"):
    key = pf.replace(" vs ", "").replace("KernelSHAP", "KS").replace("Native", "Nat")
    M(f"agree{key}", g.feat_agree.mean()); M(f"sign{key}", g.sign_agree.mean()); M(f"rank{key}", g.rank_corr.mean())
M("agreeMin", ag.feat_agree.min()); M("agreeMax", ag.feat_agree.max())
M("agreeMin", ag.feat_agree.min()); M("agreeMax", ag.feat_agree.max())
M("agreeMinWhere", f"{DSLAB[ag.loc[ag.feat_agree.idxmin(), 'dataset']]}, {ag.loc[ag.feat_agree.idxmin(), 'model']}")
# correlation between dataset max |r| and agreement
ag2 = ag.merge(pd.DataFrame([{"dataset": d, "maxr": m["max_abs_corr"], "nfeat": m["n_features"]} for d, m in metas.items()]),
               on="dataset")
from scipy.stats import spearmanr  # noqa: E402
gks = ag2[ag2.pairf == "KernelSHAP vs LIME"].groupby("dataset").agg(fa=("feat_agree", "mean"), maxr=("maxr", "first"),
                                                                  nfeat=("nfeat", "first"))
M("agreeCorrMaxr", spearmanr(gks.fa, gks.maxr).correlation, 2)
M("agreeCorrNfeat", spearmanr(gks.fa, gks.nfeat).correlation, 2)

## ROAR

In [33]:
# ---------------------------------------------------------------- ROAR
ro = load("roar.csv")
if len(ro):
    ro["family"] = ro.explainer.map(lambda a: NATIVE.get(a, a))
    ro["drop"] = ro.auroc_full - ro.auroc
    rr = ro.groupby(["model", "family", "k", "dataset"])["drop"].mean().reset_index()
    rs = rr.groupby(["model", "family", "k"])["drop"].agg(["mean", "std"]).reset_index()
    rs.to_csv(os.path.join(GEN, "roar_summary.csv"), index=False)
    fig, axs = plt.subplots(1, 2, figsize=(6.6, 2.4), sharey=True)
    for ax, k in zip(axs, ["LR", "XGB"]):
        for f_ in EXPL:
            s = rs[(rs.model == k) & (rs.family == f_)]
            ax.plot(s.k, s["mean"], marker="o", ms=4, lw=1.6, color=COL[f_],
                    label={"Native": "Native (exact linear / TreeSHAP)"}.get(f_, f_),
                    ls="--" if f_ == "Random" else "-")
        ax.set_title({"LR": "Logistic regression", "XGB": "XGBoost"}[k], color=INK)
        ax.set_xlabel("Features removed before retraining ($k$)")
        ax.set_xticks([1, 3, 5]); ax.grid(axis="y", color=GRID, lw=0.6)
    axs[0].set_ylabel("AUROC drop (mean over datasets)")
    axs[1].legend(loc="upper left", fontsize=7)
    fig.tight_layout(); fig.savefig(os.path.join(GEN, "fig_roar.pdf")); plt.close(fig)
    for k in ["LR", "XGB"]:
        for f_ in EXPL:
            v = rs[(rs.model == k) & (rs.family == f_) & (rs.k == 5)]["mean"]
            if len(v):
                M(f"roar{k}{f_}", float(v.iloc[0]))
    # per-dataset table at k = 3
    t = rr[rr.k == 3].pivot_table(index=["dataset"], columns=["model", "family"], values="drop")
    rows = []
    for d in DS:
        if d not in t.index:
            continue
        r = {"Dataset": DSLAB[d]}
        for k in ["LR", "XGB"]:
            for f_ in EXPL:
                if (k, f_) in t.columns:
                    r[f"{k} {f_[:2] if f_ != 'KernelSHAP' else 'KS'}"] = f"{t.loc[d, (k, f_)]:.3f}"
        rows.append(r)
    tex(pd.DataFrame(rows), "tab_roar.tex",
        "ROAR: test AUROC lost after removing the three globally most important features (by each explainer) and "
        "retraining with the same hyper-parameters. KS = KernelSHAP, LI = LIME, Na = native (exact linear for LR, "
        "TreeSHAP for XGB), Ra = mean of five random rankings. Larger = the ranking found more predictive features.",
        "tab:roar", size="\\scriptsize", wide=True)

## Figure: faithfulness and stability per dataset

In [34]:
# ---------------------------------------------------------------- figure: faithfulness and stability per dataset
fig, axs = plt.subplots(1, 4, figsize=(7.2, 2.9), sharey=True)
for ax, k in zip(axs, MODELS):
    for i, d in enumerate(DS):
        for j, f_ in enumerate(EXPL):
            g = im[(im.dataset == d) & (im.model == k) & (im.family == f_)].gap
            if not len(g):
                continue
            m, lo, hi = boot_mean(g)
            y = i + (j - 1.5) * 0.17
            ax.plot([lo, hi], [y, y], color=COL[f_], lw=1.2)
            ax.plot(m, y, "o", ms=3.6, color=COL[f_], mec="white", mew=0.5,
                    label={"Native": "Native"}.get(f_, f_) if (i == 0) else None)
    ax.axvline(0, color=INK2, lw=0.6)
    ax.set_title({"LR": "Logistic regression", "EBM": "EBM", "XGB": "XGBoost", "MLP": "MLP"}[k], color=INK)
    ax.grid(axis="x", color=GRID, lw=0.6)
axs[0].set_yticks(range(len(DS))); axs[0].set_yticklabels([DSLAB[d] for d in DS]); axs[0].invert_yaxis()
axs[0].set_yticks(range(len(DS))); axs[0].set_yticklabels([DSLAB[d] for d in DS]); axs[0].invert_yaxis()
axs[0].set_yticks(range(len(DS))); axs[0].set_yticklabels([DSLAB[d] for d in DS]); axs[0].invert_yaxis()
h, l = [], []
for ax in axs:
    hh, ll = ax.get_legend_handles_labels()
    for a, b in zip(hh, ll):
        if b not in l:
            h.append(a); l.append(b)
fig.supxlabel("Deletion gap (MoRF $-$ LeRF)", fontsize=8, y=0.09)
fig.legend(h, l, loc="lower center", ncol=4, bbox_to_anchor=(0.55, -0.01))
fig.tight_layout(rect=(0, 0.09, 1, 1)); fig.savefig(os.path.join(GEN, "fig_faithfulness.pdf")); plt.close(fig)
fig.tight_layout(rect=(0, 0.09, 1, 1)); fig.savefig(os.path.join(GEN, "fig_faithfulness.pdf")); plt.close(fig)
fig.tight_layout(rect=(0, 0.09, 1, 1)); fig.savefig(os.path.join(GEN, "fig_faithfulness.pdf")); plt.close(fig)
fig, axs = plt.subplots(1, 2, figsize=(6.6, 2.6), sharey=True)
for ax, metric, ttl in zip(axs, ["pert_sp", "retrain_sp"], ["Input perturbation (5% noise)", "Bootstrap retraining"]):
    for j, f_ in enumerate(EXPL[:3]):
        for i, k in enumerate(MODELS):
            b = blocks[(blocks.model == k) & (blocks.family == f_)][metric]
            if not len(b):
                continue
            x = i + (j - 1) * 0.22
            ax.scatter(np.full(len(b), x), b, s=9, color=COL[f_], alpha=0.55, lw=0)
            ax.plot([x - 0.08, x + 0.08], [b.mean()] * 2, color=COL[f_], lw=2, label=f_ if i == 0 else None)
    ax.set_xticks(range(4)); ax.set_xticklabels(MODELS); ax.set_title(ttl, color=INK)
    ax.grid(axis="y", color=GRID, lw=0.6)
axs[0].set_ylabel("Spearman of $|$attributions$|$\n(dot = dataset, bar = mean)")
axs[1].legend(loc="lower left", fontsize=7)
fig.tight_layout(); fig.savefig(os.path.join(GEN, "fig_stability.pdf")); plt.close(fig)
fig.tight_layout(); fig.savefig(os.path.join(GEN, "fig_stability.pdf")); plt.close(fig)
fig.tight_layout(); fig.savefig(os.path.join(GEN, "fig_stability.pdf")); plt.close(fig)

## Semi-synthetic

In [35]:
# ---------------------------------------------------------------- semi-synthetic
ss_f = os.path.join(RES, "semisynthetic", "semisynthetic.csv")
if os.path.exists(ss_f):
    ss = pd.read_csv(ss_f)
    ss["family"] = ss.explainer.map(lambda a: NATIVE.get(a, a))
    g = ss.groupby(["rho", "model", "family"])[["local_p5", "global_p5", "share_true", "share_proxy", "auroc"]].mean().reset_index()
    g.to_csv(os.path.join(GEN, "semisynthetic_summary.csv"), index=False)
    smods = ["LR", "EBM", "XGB", "MLP"]
    fig, axs = plt.subplots(2, 4, figsize=(7.2, 3.8), sharex=True, sharey="row")
    for c, k in enumerate(smods):
        for f_ in EXPL:
            s = g[(g.model == k) & (g.family == f_)]
            if not len(s):
                continue
            for r_, met in enumerate(["share_true", "share_proxy"]):
                axs[r_, c].plot(s.rho, s[met], marker="o", ms=3, lw=1.5, color=COL[f_],
                                ls="--" if f_ == "Random" else "-", label=f_ if (r_ == 0 and c == 0) else None)
        axs[0, c].set_title({"LR": "Logistic regression", "EBM": "EBM", "XGB": "XGBoost", "MLP": "MLP"}[k], color=INK)
        axs[1, c].set_xlabel("Proxy correlation $\\rho$")
        for r_ in range(2):
            axs[r_, c].grid(axis="y", color=GRID, lw=0.6)
    axs[0, 0].set_ylabel("Attribution share on\nthe 5 true features")
    axs[1, 0].set_ylabel("Attribution share on\ntheir 5 proxies")
    fig.legend(loc="lower center", ncol=4, bbox_to_anchor=(0.55, -0.01))
    fig.tight_layout(rect=(0, 0.06, 1, 1)); fig.savefig(os.path.join(GEN, "fig_semisynthetic.pdf")); plt.close(fig)
    rows = []
    for k in smods:
        for f_ in EXPL[:3]:
            s = g[(g.model == k) & (g.family == f_)].set_index("rho")
            if not len(s):
                continue
            rows.append({"Model": k, "Explainer": f_ if f_ != "Native" else {"LR": "Exact linear", "EBM": "EBM terms",
                                                                              "XGB": "TreeSHAP"}[k],
                         **{f"$\\rho$={r:g}": f"{s.loc[r, 'local_p5']:.2f}" for r in s.index}})
    tex(pd.DataFrame(rows), "tab_semisynthetic.tex",
        "Semi-synthetic ground truth: local precision@5 (share of each instance's top-5 features that are true "
        "generating features), by proxy correlation $\\rho$; mean of two seeds.", "tab:semisyn",
        size="\\scriptsize", wide=True)
    for k in smods:
        for f_ in EXPL[:3]:
            s = g[(g.model == k) & (g.family == f_)].set_index("rho")
            if len(s) and 0.99 in s.index:
                M(f"ssTrue{k}{f_}Zero", s.loc[0.0, "share_true"]); M(f"ssTrue{k}{f_}High", s.loc[0.99, "share_true"])
                M(f"ssPfive{k}{f_}Zero", s.loc[0.0, "local_p5"]); M(f"ssPfive{k}{f_}High", s.loc[0.99, "local_p5"])
                M(f"ssProxy{k}{f_}High", s.loc[0.99, "share_proxy"])
    M("ssAucMin", ss.auroc.min()); M("ssAucMax", ss.auroc.max())

## Subgroup (sex) analysis

In [36]:
# ---------------------------------------------------------------- subgroup (sex) analysis
sub = []
for (d, k, f_), g_ in im[(im.family != "Random") & im.group.notna()].groupby(["dataset", "model", "family"]):
    gs = sorted(g_.group.unique())
    if len(gs) != 2:
        continue
    a, b = g_[g_.group == gs[0]], g_[g_.group == gs[1]]
    rng = np.random.default_rng(0)
    for metric in ["gap", "pert_sp"]:
        diff = a[metric].mean() - b[metric].mean()
        bs = [a[metric].values[rng.integers(0, len(a), len(a))].mean() - b[metric].values[rng.integers(0, len(b), len(b))].mean()
              for _ in range(1000)]
        lo, hi = np.percentile(bs, [2.5, 97.5])
        sub.append(dict(dataset=d, model=k, family=f_, metric=metric, groups=f"{gs[0]}-{gs[1]}", n_a=len(a), n_b=len(b),
                        diff=diff, lo=lo, hi=hi, excludes_zero=(lo > 0) or (hi < 0)))
sub = pd.DataFrame(sub)
sub.to_csv(os.path.join(GEN, "subgroup.csv"), index=False)
M("subN", len(sub)); M("subSig", int(sub.excludes_zero.sum()))
M("subN", len(sub)); M("subSig", int(sub.excludes_zero.sum()))
M("subSigGap", int(sub[sub.metric == "gap"].excludes_zero.sum())); M("subNGap", int((sub.metric == "gap").sum()))
M("subSigGap", int(sub[sub.metric == "gap"].excludes_zero.sum())); M("subNGap", int((sub.metric == "gap").sum()))
M("subSigPert", int(sub[sub.metric == "pert_sp"].excludes_zero.sum())); M("subNPert", int((sub.metric == "pert_sp").sum()))
M("subSigPert", int(sub[sub.metric == "pert_sp"].excludes_zero.sum())); M("subNPert", int((sub.metric == "pert_sp").sum()))
M("subMaxAbsGap", sub[sub.metric == "gap"]["diff"].abs().max())

## Timing

In [37]:
# ---------------------------------------------------------------- timing
tm = load("timing.csv")
tm["family"] = tm.explainer.map(lambda a: NATIVE.get(a, a))
tt = tm[tm.family != "Random"].groupby(["family"]).per_instance.agg(["median", "max"])
for f_, r in tt.iterrows():
    M(f"time{f_}", r["median"], 3); M(f"timeMax{f_}", r["max"], 2)

## Global importance concordance across explainers

In [38]:
# ---------------------------------------------------------------- global importance concordance across explainers
gl = load("global_importance.csv")
gl["family"] = gl.explainer.map(lambda a: NATIVE.get(a, a))
tops = []
for (d, k), g_ in gl.groupby(["dataset", "model"]):
    t = {f_: set(x.sort_values("rank").feature.head(3)) for f_, x in g_.groupby("family")}
    fams = list(t)
    for i in range(len(fams)):
        for j in range(i + 1, len(fams)):
            tops.append(len(t[fams[i]] & t[fams[j]]) / 3)
M("globalTopThreeAgree", float(np.mean(tops)))
# write macros
with open(os.path.join(GEN, "macros.tex"), "w") as f:
    for k, v in sorted(macros.items()):
        f.write(f"\\newcommand{{\\{k}}}{{{v}}}\n")
json.dump(macros, open(os.path.join(GEN, "macros.json"), "w"), indent=1)
print(f"wrote {len(macros)} macros and tables to {GEN}")

wrote 180 macros and tables to D:\New_Research_2027\Clinical_XAI_Faithfulness\outputs\paper_generated
